# Patches — Bloc 7, Sessió 12
### Classificadors: KNN, Decision Tree, SVM

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

Avui fem servir el `features_percussio.csv` que vau construir a la Sessió 11 per entrenar i comparar tres classificadors diferents. L'objectiu no és "quin guanya" (amb només 29 mostres, qualsevol comparació d'accuracy té poca significació estadística) — l'objectiu és entendre el **procediment estàndard** de ML (train/test split, entrenar, avaluar, interpretar) i veure que **diferents algorismes "pensen" de manera diferent**, encara que arribin a conclusions semblants.

In [ ]:
# Si treballes des de Google Colab amb el repositori clonat:
import os
if not os.path.exists('/content/tp2'):
    os.system('git clone -q https://github.com/jcomajuncosas/tp2.git /content/tp2')

# El CSV de la Sessió 11 viu només a la teva sessió de Colab d'aquella classe
# (no es desa al repositori). Si encara el tens al teu Drive/sessió, puja'l i
# ajusta CSV_PATH; si no, el regenerem aquí en pocs segons a partir del mateix
# dataset/ que vau usar a la Sessió 11 — el resultat és idèntic.
DATASET_DIR = "/content/tp2/06_analisi_ml/sessio11_fft_features/dataset"
CSV_PATH = "features_percussio.csv"

if not os.path.exists(CSV_PATH):
    print("Regenerant features_percussio.csv a partir del dataset de la Sessió 11...")
    import librosa, glob, warnings
    warnings.filterwarnings('ignore')
    rows = []
    for categoria in ['kick', 'hihat']:
        for f in sorted(glob.glob(f"{DATASET_DIR}/{categoria}/*.wav")):
            audio, sr = librosa.load(f, sr=None)
            n_fft = min(2048, len(audio))
            centroid = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=n_fft).mean()
            zcr = librosa.feature.zero_crossing_rate(audio).mean()
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, n_fft=n_fft).mean(axis=1)
            row = {'fitxer': f.split('/')[-1], 'classe': categoria, 'centroid': centroid, 'zcr': zcr}
            for i, val in enumerate(mfcc):
                row[f'mfcc_{i}'] = val
            rows.append(row)
    import pandas as pd
    pd.DataFrame(rows).to_csv(CSV_PATH, index=False)
    print("Fet.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv(CSV_PATH)
print(f"Dataset: {df.shape[0]} sons, {df.shape[1]} columnes")
df.head()

## 1. Train/test split: per què cal separar les dades

Si entrenem un model amb TOTES les dades i el "comprovem" amb les mateixes dades, sempre semblarà que funciona perfectament — el model pot haver-se après les dades de memòria, no haver après el patró general (això es diu **overfitting**). Per saber si realment ha après alguna cosa útil, cal avaluar-lo amb dades que NO ha vist durant l'entrenament.

In [ ]:
X = df[['centroid', 'zcr']].values
y = (df['classe'] == 'hihat').astype(int).values  # 0 = kick, 1 = hihat

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} mostres ({np.sum(y_train==0)} kicks, {np.sum(y_train==1)} hihats)")
print(f"Test:  {len(X_test)} mostres ({np.sum(y_test==0)} kicks, {np.sum(y_test==1)} hihats)")
print()
print("'stratify=y' assegura que train i test mantenen la mateixa proporció")
print("de classes que el dataset original — important amb datasets petits.")

## 2. Tres classificadors, el mateix procediment

Cada classificador d'`sklearn` segueix sempre el mateix patró: `fit()` per entrenar, `predict()` per predir.

In [ ]:
classificadors = {
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'SVM (rbf)': SVC(kernel='rbf', random_state=42),
}

resultats = {}
for nom, clf in classificadors.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultats[nom] = (clf, y_pred, acc)
    print(f"{nom}: accuracy = {acc:.0%}")

print()
print("⚠️ Amb només 29 mostres (9 de test), aquests percentatges tenen poca")
print("significació estadística — un sol so mal classificat ja canvia 'acc'")
print("en més d'un 10%. El que importa avui és el PROCEDIMENT (com s'entrena,")
print("com s'avalua, com s'interpreta), no declarar un 'guanyador'.")

## 3. Confusion matrix: on s'equivoca (si s'equivoca)

L'accuracy és un sol número; la matriu de confusió mostra el detall — quants kicks s'han classificat bé/malament com a kick/hihat, i el mateix per als hihats.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nom, (clf, y_pred, acc)) in zip(axs, resultats.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['kick', 'hihat'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"{nom}\n(accuracy {acc:.0%})")
plt.tight_layout()
plt.show()

print("Llegir una confusion matrix: la diagonal (de dalt-esquerra a baix-dreta)")
print("són els encerts; fora de la diagonal, els errors (i de quin tipus).")

## 4. Fronteres de decisió: com "pensa" cada algorisme

Aquí ve la part més reveladora. Amb `centroid`/`zcr` el problema és tan fàcil (les classes estan molt separades) que els tres algorismes dibuixen pràcticament la mateixa frontera — una simple línia vertical. Per veure-hi diferència real, fem servir dues altres features (`mfcc_1`, `mfcc_2`), on la relació entre classes és menys òbvia.

In [ ]:
X2 = df[['mfcc_1', 'mfcc_2']].values
y2 = (df['classe'] == 'hihat').astype(int).values

classificadors2 = {
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'SVM (rbf)': SVC(kernel='rbf', random_state=42),
}

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))

x_min, x_max = X2[:, 0].min() - 20, X2[:, 0].max() + 20
y_min, y_max = X2[:, 1].min() - 20, X2[:, 1].max() + 20
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

for ax, (nom, clf) in zip(axs, classificadors2.items()):
    clf.fit(X2, y2)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn', levels=[-0.5, 0.5, 1.5])
    ax.scatter(X2[y2==0, 0], X2[y2==0, 1], c='#2a9d8f', label='kick', s=60, edgecolor='k')
    ax.scatter(X2[y2==1, 0], X2[y2==1, 1], c='#e76f51', label='hihat', s=60, edgecolor='k')
    ax.set_title(nom)
    ax.set_xlabel('MFCC 1')
    ax.set_ylabel('MFCC 2')
    ax.legend()

plt.tight_layout()
plt.show()

**Què veieu:**
- **Decision Tree**: la frontera és sempre un conjunt de talls rectes, perpendiculars als eixos (un eix cada vegada) — per això la regió té cantonades noranta graus.
- **KNN**: la frontera segueix de molt a prop els punts individuals — és "irregular", perquè cada punt nou es classifica mirant els seus veïns més propers, sense cap regla global.
- **SVM (kernel rbf)**: la frontera és una corba suau — busca el marge més ample possible entre les dues classes, no es fixa en punts individuals concrets.

**Fixeu-vos en un detall real i interessant:** si mireu el gràfic del SVM amb atenció, és possible que vegeu 1-2 punts taronja (hihat) dins la regió verda (kick) — el SVM els classifica malament! No és un error del codi: és el preu de buscar una frontera "raonable" i suau en lloc d'ajustar-se a cada punt individual. KNN i Decision Tree, en canvi, solen encertar el 100% d'aquests mateixos punts perquè es poden "doblegar" molt més lliurement per encaixar casos concrets — això no vol dir que siguin "millors", pot voler dir que estan més ajustats (potencialment massa) a aquestes 29 mostres exactes. És exactament el tipus de compromís (marge ampli i robust vs. ajust perfecte a les dades vistes) que cal saber llegir en qualsevol classificador real.

## 5. Per provar tu mateix

Canvia quin parell de features es fa servir a la Secció 4 (per exemple `mfcc_0`/`mfcc_3`, o `centroid`/`mfcc_5`) i mira com canvien les fronteres. Prova també canviar `n_neighbors` del KNN o `max_depth` de l'arbre — com afecta la forma de la frontera?

## Recordatori

Avui hem vist el procediment estàndard de classificació supervisada: dividir dades → entrenar diversos models → avaluar-los → interpretar (no només el número, també la forma de la decisió). És exactament el mateix esquelet que faríeu servir amb qualsevol altre problema de classificació, no només amb so.